1. Solar Battery Arbitrage/Configuration & Setup:

Imports core libraries (math, pandas, entsoe) and initialises the ENTSO-E client for European energy market data.
Defines countries (AT, BE, BG, CZ, FR, GR, HU, ES) with their full names for reporting.
Models a Tesla Powerwall 3 or similar with 13.5 kWh capacity, 4.6 kW discharge, and 5.0 kW charge rate.
Sets cost assumptions at €7,200 per Powerwall and €1,000 per gateway (ex-VAT, ex-installation).
Arbitrage logic targets covering 20% of average peak-hour energy, using the top-5 price hours per day.
Data range is set to the full calendar year 2024 (UTC), from 1 Jan 2024 to 1 Jan 2025.

In [1]:
import math
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
from requests import HTTPError

# If not already defined:
client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")  # <-- your token

preselected_countries = ['AT','BE','BG','CZ','FR','GR','HU','ES']

country_names = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CZ": "Czech Republic",
    "FR": "France",
    "GR": "Greece",
    "HU": "Hungary",
    "ES": "Spain",
}

# Powerwall 3 specs
pw_energy_MWh   = 13.5 / 1000.0  # 13.5 kWh
pw_discharge_MW = 4.6  / 1000.0  # 4.6 kW
pw_charge_MW    = 5.0  / 1000.0  # 5.0 kW

# Cost assumptions (excluding VAT, installation)
price_powerwall_eur = 7200.0
price_gateway_eur   = 1000.0  # one gateway per country/system

# Design choices
target_fraction = 0.20        # cover 20% of average peak-hour energy
STRONG_SOLAR_THRESHOLD = 0.20 # strong solar hours = >20% of peak solar
N_PEAK = 5                    # top-5 price hours per day

start_2024 = pd.Timestamp("2024-01-01", tz="UTC")
end_2024   = pd.Timestamp("2025-01-01", tz="UTC")

2. Mean MWh in top‑5 price hours per day (mean_peak_MWh): 

For each country, fetches day-ahead prices (resampled to hourly) and actual generation
from ENTSO-E.
Both series are inner-joined on a shared hourly index; rows with NaNs are dropped.
Days with fewer than 5 valid hours are skipped; the top-5 highest-price hours are selected from the rest.
The mean generation volume across all such peak hours in 2024 is stored in mean_peak_MWh.
Countries with missing data, API errors, or no qualifying days are logged in failed_mean.

In [2]:
mean_peak_MWh = {}   # country -> mean MWh per peak hour
failed_mean   = []

for cc in preselected_countries:
    print(f"\n=== {cc} ===")
    try:
        # Prices (2024, hourly)
        prices = client.query_day_ahead_prices(country_code=cc,
                                               start=start_2024,
                                               end=end_2024)
        if prices.empty:
            print("No price data")
            failed_mean.append(cc)
            continue
        prices_hourly = prices.resample("1h").mean()

        # Generation: total MW (sum of all Actual Aggregated technologies)
        gen = client.query_generation(cc, start=start_2024, end=end_2024)
        if gen.empty:
            print("No generation data")
            failed_mean.append(cc)
            continue

        if isinstance(gen.columns, pd.MultiIndex):
            mask = gen.columns.get_level_values(1) == "Actual Aggregated"
            gen_agg = gen.loc[:, mask]
        else:
            gen_agg = gen
        total_gen_MW_15min = gen_agg.sum(axis=1)  # MW at 15-min

        # 15-min → hourly MWh: sum MW over 4 intervals * 0.25 h
        total_gen_MWh_hourly = total_gen_MW_15min.resample("1h").sum() * 0.25

        # Align prices and volume
        df = pd.DataFrame(index=prices_hourly.index)
        df["price_EUR_MWh"] = prices_hourly
        df = df.join(total_gen_MWh_hourly.to_frame(name="volume_MWh"), how="inner")
        df = df.dropna()
        if df.empty:
            print("No overlapping price/volume data")
            failed_mean.append(cc)
            continue

        df["date"] = df.index.date

        peak_volumes = []
        for date, day_df in df.groupby("date"):
            if len(day_df) < N_PEAK:
                continue
            top5 = day_df.sort_values("price_EUR_MWh", ascending=False).head(N_PEAK)
            peak_volumes.append(top5["volume_MWh"])

        if not peak_volumes:
            print("No peak hours found")
            failed_mean.append(cc)
            continue

        peak_all = pd.concat(peak_volumes)
        mean_MWh_peak_5 = peak_all.mean()

        mean_peak_MWh[cc] = mean_MWh_peak_5
        print(f"Mean MWh in top-5 price hours (2024): {mean_MWh_peak_5:,.0f} MWh/hour")

    except (NoMatchingDataError, HTTPError, Exception) as e:
        print("Error:", e)
        failed_mean.append(cc)

print("\nCountries where mean peak MWh could not be computed:", failed_mean)


=== AT ===
Mean MWh in top-5 price hours (2024): 8,498 MWh/hour

=== BE ===
Mean MWh in top-5 price hours (2024): 2,007 MWh/hour

=== BG ===
Mean MWh in top-5 price hours (2024): 1,058 MWh/hour

=== CZ ===
Mean MWh in top-5 price hours (2024): 5,263 MWh/hour

=== FR ===
Mean MWh in top-5 price hours (2024): 17,316 MWh/hour

=== GR ===
Mean MWh in top-5 price hours (2024): 1,554 MWh/hour

=== HU ===
Mean MWh in top-5 price hours (2024): 3,407 MWh/hour

=== ES ===
Mean MWh in top-5 price hours (2024): 28,602 MWh/hour

Countries where mean peak MWh could not be computed: []
